# Finora AI — QR Code & Bank Slip Scan Diagnostics (Local + Colab)

Notebook นี้จำลอง Pipeline การสแกน QR / สลิปโอนเงินของ **Finora** แบบครบทั้ง 4 ขั้นตอน เพื่อแยกแยะว่าปัญหาการอ่านผิดมาจากจุดไหน:
1. **Stage 1 — QR Decoder (`src/lib/qr-decoder.ts`)**: เตรียมภาพส่ง Vision (`<= 2MB` ส่งตรง, ใหญ่กว่าย่อ `2048px`) + ถอดรหัส QR หลาย pass (Full, High-Contrast, มุมขวาล่าง Mini-QR, ครึ่งล่าง, กลางภาพ)
2. **Stage 2 — PromptPay EMVCo TLV Parser (`src/lib/seed.ts`)**: แกะ Tag `29/30` (PromptPay ID), `54` (ยอดเงิน), `59` (ชื่อร้าน) และแยกชนิด `promptpay_emvco` vs `slip_mini_qr`
3. **Stage 3 — Vision LLM Call (`src/app/api/ai/qr/route.ts` & `src/lib/prompts.ts`)**: ส่งรูป (`detail: "high"`, `temperature: 0`, `json_object`) + Prompt v2.5.0 ไปที่ OpenRouter แล้วดู Raw JSON + Latency + Token Usage
4. **Stage 4 — Post-processing & Merge (`parseQrSlipResult` & `ai-assistant/page.tsx`)**: ตรวจว่า `clean_slip_name`, `userName` matching, ตัวเลข/หมวดหมู่, หรือการทับค่าด้วย TLV ทำให้ผลลัพธ์เพี้ยนหรือไม่

### วิธีใช้งานบนเครื่อง (VS Code / Jupyter)
1. นำรูปสลิป/ป้าย QR ที่ต้องการเทสใส่ในโฟลเดอร์ `tests/llm/samples/` (โฟลเดอร์นี้ถูก gitignore ไว้แล้ว จะไม่หลุดเข้า Git)
2. Notebook จะอ่าน `OPENROUTER_API_KEY` และ `OPENROUTER_MODEL` จาก `.env.local` ที่ root โปรเจกต์ให้อัตโนมัติ
3. กด **Run All** ได้ทันที — หรือถ้าเปิดบน Google Colab สามารถตั้ง `OPENROUTER_API_KEY` ใน Secrets และอัปโหลดรูปผ่านหน้าเว็บใน Cell 4 ได้เช่นกัน


In [ ]:
import importlib
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        ["bash", "-lc", "apt-get -qq update && apt-get -qq install -y libzbar0"],
        check=False,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install", "pyzbar", "pillow", "opencv-python-headless", "requests", "pandas"],
        check=True,
    )

_required = {
    "PIL": "pillow",
    "cv2": "opencv-python-headless",
    "numpy": "numpy",
    "requests": "requests",
    "pandas": "pandas",
}
_missing = []
for _mod, _pkg in _required.items():
    try:
        importlib.import_module(_mod)
    except Exception:
        _missing.append(_pkg)

_has_pyzbar = False
try:
    importlib.import_module("pyzbar")
    _has_pyzbar = True
except Exception:
    _has_pyzbar = False

print(f"Environment: {'Google Colab' if IN_COLAB else f'Local ({sys.platform})'}")
if _missing:
    print(f"Missing required packages: {', '.join(_missing)} -> run: pip install {' '.join(_missing)}")
else:
    print("All required Python packages installed (PIL, cv2, numpy, requests, pandas).")
if not _has_pyzbar:
    print("Optional pyzbar not loaded: using OpenCV QRCodeDetector() fallback (works on Windows without libzbar).")

In [ ]:
import getpass
import os
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".env.local").is_file() or (candidate / "package.json").is_file():
            return candidate
    return start


def _read_env_file(path: Path) -> dict[str, str]:
    env_map: dict[str, str] = {}
    if not path.is_file():
        return env_map
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        k = k.strip()
        v = v.strip()
        if len(v) >= 2 and ((v[0] == v[-1] == '"') or (v[0] == v[-1] == "'")):
            v = v[1:-1]
        env_map[k] = v
    return env_map


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
_local_env = _read_env_file(REPO_ROOT / ".env.local")

_secret_key = ""
try:
    from google.colab import userdata
    _secret_key = (userdata.get("OPENROUTER_API_KEY") or "").strip()
except Exception:
    _secret_key = ""

OPENROUTER_API_KEY = (
    _secret_key
    or os.environ.get("OPENROUTER_API_KEY", "").strip()
    or _local_env.get("OPENROUTER_API_KEY", "").strip()
    or getpass.getpass("กรอก OpenRouter API Key (sk-or-v1-...): ").strip()
)

# ค่าตั้งต้น: อ่าน OPENROUTER_MODEL จาก env/.env.local ก่อน แล้วค่อย fallback เป็น google/gemini-2.5-flash
MODEL = (
    os.environ.get("OPENROUTER_MODEL", "").strip()
    or _local_env.get("OPENROUTER_MODEL", "").strip()
    or "google/gemini-2.5-flash"
)
USER_NAME = "คุณธนภัทร วงศ์เจริญ"
IMAGE_DETAIL = "high"  # "high" | "low" | "auto"
TEMPERATURE = 0.0

# รายการหมวดหมู่ตรงกับ INITIAL_CATEGORIES ใน src/lib/seed.ts (ตัด "recurring" ออก)
CATEGORIES = [
    {"id": "food", "name": "อาหาร & เครื่องดื่ม", "type": "expense"},
    {"id": "shopping", "name": "ช้อปปิ้ง & ไลฟ์สไตล์", "type": "expense"},
    {"id": "housing", "name": "ที่อยู่อาศัย & บิล", "type": "expense"},
    {"id": "transport", "name": "การเดินทาง & น้ำมัน", "type": "expense"},
    {"id": "entertainment", "name": "ความบันเทิง & สมาชิก", "type": "expense"},
    {"id": "salary", "name": "เงินเดือน & โบนัส", "type": "income"},
    {"id": "investment", "name": "เงินปันผล & การลงทุน", "type": "income"},
]

print(f"REPO_ROOT={REPO_ROOT}")
print(f"MODEL={MODEL} | detail={IMAGE_DETAIL} | temperature={TEMPERATURE} | user={USER_NAME}")
print(f"API key loaded: {'Yes' if OPENROUTER_API_KEY else 'No'}")

In [ ]:
import base64
import io
import json
import math
import mimetypes
import re
import time
from typing import Any, Callable, Optional

import cv2
import numpy as np
import requests
from PIL import Image, ImageOps

try:
    from pyzbar.pyzbar import decode as _pyzbar_decode
except Exception:
    _pyzbar_decode = None

DIRECT_UPLOAD_MAX_BYTES = 2 * 1024 * 1024
DIRECT_MIME_TYPES = {"image/png", "image/jpeg", "image/webp"}
_PIL_FORMAT_TO_MIME = {
    "PNG": "image/png",
    "JPEG": "image/jpeg",
    "JPG": "image/jpeg",
    "WEBP": "image/webp",
}


def _detect_mime(filename: str, img: Image.Image) -> str:
    fmt = (img.format or "").upper()
    if fmt in _PIL_FORMAT_TO_MIME:
        return _PIL_FORMAT_TO_MIME[fmt]
    guessed, _ = mimetypes.guess_type(filename)
    return guessed or "image/jpeg"


def _composite_white(pil_img: Image.Image) -> Image.Image:
    if pil_img.mode in ("RGBA", "LA") or (pil_img.mode == "P" and "transparency" in pil_img.info):
        rgba = pil_img.convert("RGBA")
        bg = Image.new("RGB", rgba.size, (255, 255, 255))
        bg.paste(rgba, mask=rgba.split()[3])
        return bg
    return pil_img.convert("RGB")


def _decode_qr_from_pil(pil_rgb: Image.Image) -> Optional[str]:
    if _pyzbar_decode is not None:
        try:
            hits = _pyzbar_decode(pil_rgb)
            for h in hits:
                val = h.data.decode("utf-8", errors="replace").strip()
                if val:
                    return val
        except Exception:
            pass

    detector = cv2.QRCodeDetector()
    arr = np.array(pil_rgb)
    for variant in (arr, 255 - arr):
        try:
            data, _, _ = detector.detectAndDecode(variant)
            if data and data.strip():
                return data.strip()
        except Exception:
            pass
    return None


def _run_qr_pass(
    pil_white: Image.Image,
    sx: float,
    sy: float,
    sw: float,
    sh: float,
    max_dim: int,
    high_contrast: bool,
) -> Optional[str]:
    iw, ih = pil_white.size
    x0 = max(0, min(iw - 1, int(round(sx))))
    y0 = max(0, min(ih - 1, int(round(sy))))
    x1 = max(x0 + 1, min(iw, int(round(sx + sw))))
    y1 = max(y0 + 1, min(ih, int(round(sy + sh))))
    crop = pil_white.crop((x0, y0, x1, y1))

    cw, ch = crop.size
    scale = min(1.0, max_dim / max(cw, ch))
    w = max(1, int(round(cw * scale)))
    h = max(1, int(round(ch * scale)))
    if w != cw or h != ch:
        crop = crop.resize((w, h), Image.Resampling.LANCZOS)

    if high_contrast:
        arr = np.asarray(crop, dtype=np.float32)
        lum = 0.299 * arr[:, :, 0] + 0.587 * arr[:, :, 1] + 0.114 * arr[:, :, 2]
        binary = np.where(lum < 140, 0, 255).astype(np.uint8)
        crop = Image.fromarray(np.stack([binary, binary, binary], axis=-1), mode="RGB")

    return _decode_qr_from_pil(crop)


def decode_qr_image(image_bytes: bytes, filename: str = "slip.jpg") -> dict[str, Any]:
    """Mirror src/lib/qr-decoder.ts: prepare base64 for Vision + multi-pass QR decode."""
    raw_img = ImageOps.exif_transpose(Image.open(io.BytesIO(image_bytes)))
    mime_type = _detect_mime(filename, raw_img)
    iw, ih = raw_img.size

    can_send_direct = (
        len(image_bytes) <= DIRECT_UPLOAD_MAX_BYTES
        and mime_type in DIRECT_MIME_TYPES
        and len(image_bytes) > 0
    )

    white_img = _composite_white(raw_img)
    if can_send_direct:
        image_base64 = base64.b64encode(image_bytes).decode("ascii")
        sent_direct = True
    else:
        v_scale = min(1.0, 2048.0 / max(iw, ih))
        vw = max(1, int(round(iw * v_scale)))
        vh = max(1, int(round(ih * v_scale)))
        resized = white_img.resize((vw, vh), Image.Resampling.LANCZOS) if (vw != iw or vh != ih) else white_img
        buf = io.BytesIO()
        resized.save(buf, format="JPEG", quality=95)
        image_base64 = base64.b64encode(buf.getvalue()).decode("ascii")
        mime_type = "image/jpeg"
        sent_direct = False

    passes: list[tuple[str, Callable[[], Optional[str]]]] = [
        ("native_full", lambda: _decode_qr_from_pil(white_img)),
        ("full_1200_normal", lambda: _run_qr_pass(white_img, 0, 0, iw, ih, 1200, False)),
        ("full_1800_high_contrast", lambda: _run_qr_pass(white_img, 0, 0, iw, ih, 1800, True)),
        ("bottom_right_normal", lambda: _run_qr_pass(white_img, iw * 0.45, ih * 0.55, iw * 0.55, ih * 0.45, 1200, False)),
        ("bottom_right_high_contrast", lambda: _run_qr_pass(white_img, iw * 0.45, ih * 0.55, iw * 0.55, ih * 0.45, 1200, True)),
        ("bottom_half_normal", lambda: _run_qr_pass(white_img, 0, ih * 0.5, iw, ih * 0.5, 1400, False)),
        ("bottom_half_high_contrast", lambda: _run_qr_pass(white_img, 0, ih * 0.5, iw, ih * 0.5, 1400, True)),
        ("center_crop_normal", lambda: _run_qr_pass(white_img, iw * 0.15, ih * 0.15, iw * 0.7, ih * 0.7, 1400, False)),
        ("center_crop_high_contrast", lambda: _run_qr_pass(white_img, iw * 0.15, ih * 0.15, iw * 0.7, ih * 0.7, 1400, True)),
    ]

    qr_raw: Optional[str] = None
    matched_pass: Optional[str] = None
    for pass_name, fn in passes:
        hit = fn()
        if hit:
            qr_raw = hit
            matched_pass = pass_name
            break

    return {
        "qr_raw": qr_raw,
        "matched_pass": matched_pass,
        "image_base64": image_base64,
        "mime_type": mime_type,
        "width": iw,
        "height": ih,
        "file_bytes": len(image_bytes),
        "sent_direct": sent_direct,
    }


def parse_promptpay_payload(raw: Optional[str]) -> dict[str, Any]:
    """Mirror parsePromptPayPayload in src/lib/seed.ts + classify QR kind."""
    out: dict[str, Any] = {}
    if not raw:
        out["qr_kind"] = "none"
        return out

    if raw.startswith("00410006000001"):
        out["isPromptPayOrSlip"] = True
        out["qr_kind"] = "slip_mini_qr"
    elif raw.startswith("000201"):
        out["isPromptPayOrSlip"] = True
        out["qr_kind"] = "promptpay_emvco"
    else:
        out["qr_kind"] = "other_qr"

    i = 0
    while i + 4 <= len(raw):
        tag = raw[i : i + 2]
        len_str = raw[i + 2 : i + 4]
        if not len_str.isdigit():
            break
        length = int(len_str)
        if length < 0 or i + 4 + length > len(raw):
            break
        val = raw[i + 4 : i + 4 + length]
        if tag == "54":
            try:
                num = float(val)
                if num > 0:
                    out["amount"] = num
            except ValueError:
                pass
        if tag == "59" and val.strip() and val.strip().upper() != "TH":
            out["merchant"] = val.strip()
        if tag in ("29", "30"):
            out["isPromptPayOrSlip"] = True
            j = 0
            while j + 4 <= len(val):
                sub_tag = val[j : j + 2]
                sub_len_str = val[j + 2 : j + 4]
                if not sub_len_str.isdigit():
                    break
                sub_len = int(sub_len_str)
                if sub_len < 0 or j + 4 + sub_len > len(val):
                    break
                sub_val = val[j + 4 : j + 4 + sub_len]
                if tag == "29" and sub_tag in ("01", "02", "03"):
                    out["promptPayId"] = re.sub(r"^0066(\d{9})$", r"0\1", sub_val)
                elif tag == "30" and sub_tag == "01":
                    out["promptPayId"] = sub_val
                j += 4 + sub_len
        i += 4 + length
    return out


def build_qr_slip_prompt(
    qr_raw: Optional[str],
    category_list: str,
    user_name: Optional[str] = None,
) -> str:
    """Mirror buildQrSlipPrompt (v2.5.0) in src/lib/prompts.ts verbatim."""
    clean_user = (user_name or "").strip()
    lines = [
        "วิเคราะห์รูปภาพนี้ว่าเป็นป้าย QR Code รับเงิน (PromptPay / Thai QR Payment) หรือสลิปธุรกรรมธนาคาร/e-Wallet หรือไม่",
        f"ข้อมูลดิบที่อ่านได้จากบาร์โค้ด QR: {qr_raw}"
        if qr_raw
        else "ไม่พบบาร์โค้ดจากตัวอ่านเบื้องต้น ให้ตรวจสอบจากภาพโดยตรง",
        f'ชื่อผู้ใช้เจ้าของบัญชีในระบบ: "{clean_user}"' if clean_user else "",
        f"หมวดหมู่ที่เลือกได้: {category_list}",
        "ตอบกลับเป็น JSON เท่านั้นตามโครงสร้างนี้:",
        '{ "isValidPaymentImage": boolean, "type": "expense" | "income", "sender": string, "receiver": string, "merchant": string, "promptPayId": string, "amount": number, "categoryId": string }',
        "กฎสำคัญสำหรับสลิปและป้าย QR หลายฟอร์แมต:",
        "1. ถ้ารูปนี้ไม่ใช่ QR Code และไม่ใช่สลิปโอนเงิน/จ่ายบิลหรือป้ายรับเงิน ให้ตั้ง isValidPaymentImage = false",
        '2. แยกผู้โอน (sender) และผู้รับ/ร้านค้า (receiver) ตามโครงสร้างสลิปแต่ละแบบ:',
        '   - กรณีสลิปโอนเงินธนาคาร (เรียง บน->ล่าง เช่น K+, SCB, KTB, BBL หรือ ซ้าย->ขวา เช่น Krungsri, TTB): sender คือฝั่งต้นทาง/ผู้โอน, receiver คือฝั่งปลายทาง/ผู้รับหรือร้านค้า',
        '   - กรณีสลิป e-Wallet / ใบเสร็จชำระบริการที่ไม่มีชื่อผู้โอนและไม่มีลูกศร (เช่น TrueMoney ที่มีชื่อร้านค้า/บริการและยอดเงินอยู่ด้านบน ตามด้วย "ช่องทางการชำระเงิน: วอลเล็ท", "Biller ID", "Ref 1/2/3"): ให้ตั้ง sender = "" และใช้ชื่อร้านค้า/บริการด้านบนเป็น receiver (ห้ามเอาชื่อแอปเช่น "truemoney", "วอลเล็ท" หรือ "สถานที่ทำรายการ" มาเป็นชื่อผู้โอน/ร้านค้า)',
        '   - กรณีป้าย QR ร้านค้า: ให้ตั้ง sender = "" และใช้ชื่อร้านบนป้ายเป็น receiver',
        '   - หากชื่อร้านค้าขึ้นบรรทัดใหม่ (เช่น บรรทัดแรกมีชื่อร้านไม่จบแล้วต่ออีกบรรทัดเหนือชื่อบุคคล/เลขบัญชี) ให้เชื่อมทุกบรรทัดของชื่อร้านเข้าด้วยกันให้ครบ (เช่น "Potato shake & ข้าวหมู" + "กรอบ" = "Potato shake & ข้าวหมูกรอบ")',
        '   - อ่านตัวอักษรภาษาไทยตามรูปอักษรที่เห็นจริงอย่างระมัดระวัง ห้ามอ่านลายน้ำพื้นหลังมาปน และห้ามเดาหรือแปลความหมายจากคำภาษาอังกฤษข้างหน้า (เช่น ห้ามเดาคำว่า "มันฝรั่ง" จากคำว่า "Potato")',
        '   - หากชื่อท้ายบรรทัดถูกแอปธนาคารตัดคำด้วย "..." (เช่น "นาย กิตติ แ...") ให้ตัดเศษตัวอักษรย่อและ "..." ท้ายชื่อออก',
        '3. การตัดสิน type และ merchant:',
        '   - หากมีชื่อผู้ใช้ในระบบ และ receiver ตรงหรือคล้ายกับชื่อผู้ใช้ (ส่วน sender เป็นคนอื่น) ให้ตั้ง type = "income" และ merchant = sender',
        '   - กรณีอื่นทั้งหมด (ผู้โอนตรงกับชื่อผู้ใช้, หรือไม่ตรงทั้งคู่, หรือเป็นป้าย QR ร้านค้า) ให้ตั้ง type = "expense" และ merchant = receiver (ถ้าไม่มี receiver ค่อยใช้ sender)',
        '4. promptPayId: เลขพร้อมเพย์ เบอร์โทร Biller ID หรือรหัสอ้างอิง/เลขที่รายการที่ปรากฏในภาพ (ถ้าไม่มีให้ส่ง "")',
        '5. amount: ยอดเงินที่โอนหรือชำระสุทธิเป็นตัวเลขเท่านั้น (ห้ามใช้ยอดค่าธรรมเนียม 0.00 และห้ามเอาเลขที่อ้างอิงมาเป็นยอดเงิน; ถ้าเป็นป้าย QR ที่ไม่ได้ระบุยอดเงิน ให้ส่ง 0 เสมอ)',
        '6. categoryId: เลือก id หมวดหมู่ที่ตรงกับ type และชื่อร้าน/บันทึกช่วยจำในสลิปที่สุด (รายรับใช้ "salary" หรือ "investment", รายจ่ายถ้าไม่แน่ใจใช้ "food" ห้ามเลือก "recurring")',
    ]
    return "\n".join([ln for ln in lines if ln])


def clean_slip_name(raw: Any) -> str:
    if not isinstance(raw, str):
        return ""
    s = raw.strip()
    s = re.sub(r"(\.{2,}|…)+$", "", s)
    s = re.sub(r"\s+[\u0E01-\u0E4E]\.?$", "", s)
    s = re.sub(r"[\u0E40-\u0E44]$", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalize_for_match(name: str) -> str:
    s = name.lower()
    s = re.sub(
        r"^(คุณ|นาย|นางสาว|นาง|น\.ส\.|ด\.ช\.|ด\.ญ\.|mr\.?|mrs\.?|ms\.?|miss)\s*",
        "",
        s,
        flags=re.IGNORECASE,
    )
    return re.sub(r"\s+", " ", s).strip()


def names_likely_match(extracted: str, user_name: Optional[str]) -> bool:
    if not extracted or not user_name:
        return False
    a = normalize_for_match(extracted)
    b = normalize_for_match(user_name)
    if not a or not b:
        return False
    if b in a or a in b:
        return True
    a_first = a.split(" ")[0]
    b_first = b.split(" ")[0]
    return len(a_first) >= 3 and a_first == b_first


def parse_qr_slip_result(
    raw_text: str,
    valid_category_ids: Optional[list[str]] = None,
    user_name: Optional[str] = None,
) -> dict[str, Any]:
    """Mirror parseQrSlipResult in src/lib/prompts.ts."""
    try:
        cleaned = re.sub(r"^```(?:json)?\s*", "", str(raw_text or ""), flags=re.IGNORECASE)
        cleaned = re.sub(r"\s*```$", "", cleaned).strip()
        parsed = json.loads(cleaned or "{}")
        if not isinstance(parsed, dict):
            return {"ok": False, "error": "ผลลัพธ์จาก AI ไม่ใช่รูปแบบ JSON Object"}

        sender = clean_slip_name(parsed.get("sender"))
        receiver = clean_slip_name(parsed.get("receiver"))
        raw_merchant = clean_slip_name(parsed.get("merchant"))

        tx_type = "income" if parsed.get("type") == "income" else "expense"
        if (user_name or "").strip() and (sender or receiver):
            sender_is_user = names_likely_match(sender, user_name)
            receiver_is_user = names_likely_match(receiver, user_name)
            if receiver_is_user and not sender_is_user and sender:
                tx_type = "income"
            elif sender_is_user and not receiver_is_user and receiver:
                tx_type = "expense"

        merchant = (
            (sender or raw_merchant or receiver)
            if tx_type == "income"
            else (receiver or raw_merchant or sender)
        )

        raw_amt = parsed.get("amount")
        try:
            if isinstance(raw_amt, (int, float)) and not isinstance(raw_amt, bool):
                raw_amt_val = float(raw_amt)
            else:
                num_str = re.sub(r"[^0-9.-]", "", str(raw_amt or ""))
                raw_amt_val = float(num_str) if num_str not in ("", "-", ".", "-.") else 0.0
        except ValueError:
            raw_amt_val = 0.0
        amount = raw_amt_val if (math.isfinite(raw_amt_val) and raw_amt_val > 0) else 0

        raw_cat = parsed.get("categoryId").strip() if isinstance(parsed.get("categoryId"), str) else ""
        default_cat = "salary" if tx_type == "income" else "food"
        category_id = (
            raw_cat
            if (
                raw_cat
                and raw_cat != "recurring"
                and (not valid_category_ids or raw_cat in valid_category_ids)
            )
            else default_cat
        )

        data: dict[str, Any] = {
            "isValidPaymentImage": bool(parsed.get("isValidPaymentImage")),
            "type": tx_type,
            "merchant": merchant,
            "promptPayId": parsed.get("promptPayId").strip()
            if isinstance(parsed.get("promptPayId"), str)
            else "",
            "amount": amount,
            "categoryId": category_id,
        }
        if sender:
            data["sender"] = sender
        if receiver:
            data["receiver"] = receiver

        return {"ok": True, "data": data, "raw_parsed": parsed}
    except Exception:
        return {"ok": False, "error": "ไม่สามารถแปลงผลลัพธ์ JSON จาก AI ได้"}


def guess_category_id(text: str, tx_type: str) -> str:
    """Mirror guessCategoryId in src/app/(app)/ai-assistant/page.tsx."""
    if tx_type == "income":
        return "investment" if re.search(r"ปันผล|หุ้น|กองทุน|ดอกเบี้ย|invest|dividend", text, re.I) else "salary"
    lower = text.lower()
    if re.search(r"bts|mrt|grab|taxi|oil|shell|ptt|เดินทาง|รถ", lower):
        return "transport"
    if re.search(r"uniqlo|apple|mall|shop|lazada|shopee|central", lower):
        return "shopping"
    if re.search(r"netflix|spotify|youtube|game|movie", lower):
        return "entertainment"
    if re.search(r"condo|rent|ais|true|water|electric|บิล|ค่าไฟ|ค่าเช่า", lower):
        return "housing"
    return "food"


def call_openrouter_qr(
    image_base64: str,
    mime_type: str,
    prompt: str,
    model: Optional[str] = None,
    api_key: Optional[str] = None,
    detail: Optional[str] = None,
    temperature: Optional[float] = None,
) -> dict[str, Any]:
    key = (api_key if api_key is not None else OPENROUTER_API_KEY).strip()
    mdl = (model or MODEL).strip()
    dtl = (detail or IMAGE_DETAIL).strip()
    temp = TEMPERATURE if temperature is None else temperature

    if not key:
        return {"ok": False, "status": 400, "error": "ยังไม่ได้ตั้งค่า OPENROUTER_API_KEY"}

    t0 = time.perf_counter()
    try:
        res = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {key}",
                "Content-Type": "application/json",
                "HTTP-Referer": "http://localhost:3000",
                "X-Title": "Finora AI Wealth Engine (Eval)",
            },
            json={
                "model": mdl,
                "temperature": temp,
                "response_format": {"type": "json_object"},
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:{mime_type or 'image/jpeg'};base64,{image_base64}",
                                    "detail": dtl,
                                },
                            },
                        ],
                    }
                ],
            },
            timeout=90,
        )
        latency_ms = int(round((time.perf_counter() - t0) * 1000))
        data = res.json()
        if not res.ok:
            err_msg = (data.get("error") or {}).get("message") or f"OpenRouter API Error ({res.status_code})"
            return {"ok": False, "status": res.status_code, "error": err_msg, "latency_ms": latency_ms}

        msg = (data.get("choices") or [{}])[0].get("message") or {}
        text = msg.get("content") if isinstance(msg.get("content"), str) else ""
        if not text.strip():
            return {"ok": False, "status": 502, "error": "AI ไม่ได้ส่งข้อความตอบกลับ", "latency_ms": latency_ms}

        usage = data.get("usage") or {}
        return {
            "ok": True,
            "status": 200,
            "text": text,
            "resolved_model": str(data.get("model") or mdl),
            "input_tokens": int(usage.get("prompt_tokens") or 0),
            "output_tokens": int(usage.get("completion_tokens") or 0),
            "total_tokens": int(usage.get("total_tokens") or 0),
            "latency_ms": latency_ms,
        }
    except Exception as exc:
        return {
            "ok": False,
            "status": 500,
            "error": str(exc),
            "latency_ms": int(round((time.perf_counter() - t0) * 1000)),
        }


def run_full_pipeline(
    image_bytes: bytes,
    filename: str = "slip.jpg",
    model: Optional[str] = None,
    user_name: Optional[str] = None,
    detail: Optional[str] = None,
    temperature: Optional[float] = None,
    custom_prompt_fn: Optional[Callable[[Optional[str], str, Optional[str]], str]] = None,
) -> dict[str, Any]:
    uname = USER_NAME if user_name is None else user_name
    filtered_cats = [c for c in CATEGORIES if c["id"] != "recurring"]
    category_list = ", ".join(f"{c['id']} ({c['type']}: {c['name']})" for c in filtered_cats)
    valid_cat_ids = [c["id"] for c in filtered_cats]

    # Stage 1: QR decode & image preparation
    stage1 = decode_qr_image(image_bytes, filename)
    qr_raw = stage1["qr_raw"]

    # Stage 2: PromptPay EMVCo TLV parse
    stage2 = parse_promptpay_payload(qr_raw)

    # Stage 3: Vision LLM call
    prompt_fn = custom_prompt_fn or build_qr_slip_prompt
    prompt = prompt_fn(qr_raw, category_list, uname)
    stage3 = call_openrouter_qr(
        image_base64=stage1["image_base64"],
        mime_type=stage1["mime_type"],
        prompt=prompt,
        model=model,
        detail=detail,
        temperature=temperature,
    )

    # Stage 4: Parse + Frontend Merge (matching src/app/(app)/ai-assistant/page.tsx)
    stage4_parse = (
        parse_qr_slip_result(stage3["text"], valid_cat_ids, uname)
        if stage3.get("ok") and stage3.get("text")
        else {"ok": False, "error": stage3.get("error", "ไม่ได้รับข้อความจาก AI")}
    )

    vision = stage4_parse.get("data") if stage4_parse.get("ok") else None
    merged: Optional[dict[str, Any]] = None
    if qr_raw or (vision and vision.get("isValidPaymentImage")):
        tx_type = "income" if (vision and vision.get("type") == "income") else "expense"
        merchant = (
            stage2.get("merchant")
            or (vision.get("merchant") if vision else "")
            or ("ร้านค้า PromptPay" if stage2.get("isPromptPayOrSlip") else ((qr_raw or "")[:40] or "รายการสแกน QR"))
        )
        promptpay_id = (
            stage2.get("promptPayId")
            or (vision.get("promptPayId") if vision else "")
            or ((qr_raw or "")[:20] if qr_raw else "-")
        )
        tlv_amt = stage2.get("amount")
        vis_amt = vision.get("amount") if vision else None
        if isinstance(tlv_amt, (int, float)) and tlv_amt > 0:
            amount = tlv_amt
        elif isinstance(vis_amt, (int, float)) and vis_amt > 0:
            amount = vis_amt
        else:
            amount = 0

        matching_cats = [c["id"] for c in filtered_cats if c["type"] == tx_type]
        vis_cat = vision.get("categoryId") if vision else ""
        if vis_cat and vis_cat in matching_cats:
            valid_category = vis_cat
            used_cat_fallback = False
        else:
            valid_category = guess_category_id(merchant, tx_type)
            used_cat_fallback = True

        merged = {
            "type": tx_type,
            "merchant": merchant,
            "sender": vision.get("sender", "") if vision else "",
            "receiver": vision.get("receiver", "") if vision else "",
            "promptPayId": promptpay_id,
            "amount": amount,
            "categoryId": valid_category,
            "usedCategoryFallback": used_cat_fallback,
        }

    # Diagnostic checklist
    diagnostics: list[str] = []
    if not stage1["sent_direct"]:
        diagnostics.append(f"INFO: ภาพถูกย่อและบีบอัดเป็น JPEG 95% (ขนาดไฟล์เดิม {stage1['file_bytes']} bytes > 2MB หรือไม่ใช่ PNG/JPEG/WEBP)")
    if not qr_raw:
        diagnostics.append("WARN [Stage 1]: ถอดรหัส QR Barcode ไม่สำเร็จ (qr_raw=None) — ระบบพึ่งพา Vision LLM 100%")
    else:
        diagnostics.append(f"OK [Stage 1]: อ่าน QR สำเร็จใน pass '{stage1['matched_pass']}'")

    if stage2["qr_kind"] == "slip_mini_qr":
        diagnostics.append("INFO [Stage 2]: พบ Mini-QR ตรวจสลิปธนาคาร (00410006...) ซึ่งไม่มี Tag 54/59 — ชื่อร้านและยอดเงินต้องอ่านจากภาพด้วย Vision LLM")
    elif stage2["qr_kind"] == "promptpay_emvco":
        if not stage2.get("merchant") or not stage2.get("amount"):
            diagnostics.append("WARN [Stage 2]: PromptPay QR ขาด Tag 59 (ชื่อร้าน) หรือ Tag 54 (ยอดเงิน) — ต้องอาศัย Vision LLM เสริม")

    if not stage3.get("ok"):
        diagnostics.append(f"FAIL [Stage 3]: เรียก OpenRouter API ไม่สำเร็จ — {stage3.get('error')}")
    elif not stage4_parse.get("ok"):
        diagnostics.append(f"FAIL [Stage 4]: แปลง JSON จาก LLM ไม่สำเร็จ — {stage4_parse.get('error')}")
    else:
        raw_p = stage4_parse.get("raw_parsed", {})
        if isinstance(raw_p.get("receiver"), str) and raw_p["receiver"].strip() != vision.get("receiver", ""):
            diagnostics.append(f"INFO [Stage 4]: clean_slip_name แก้ไขชื่อ receiver จาก {raw_p['receiver']!r} -> {vision.get('receiver', '')!r}")
        if isinstance(raw_p.get("sender"), str) and raw_p["sender"].strip() != vision.get("sender", ""):
            diagnostics.append(f"INFO [Stage 4]: clean_slip_name แก้ไขชื่อ sender จาก {raw_p['sender']!r} -> {vision.get('sender', '')!r}")
        if stage2.get("merchant") and vision.get("merchant") and stage2["merchant"] != vision["merchant"]:
            diagnostics.append(f"INFO [Merge]: ชื่อจาก QR TLV ({stage2['merchant']!r}) ถูกใช้แทนชื่อจาก Vision ({vision['merchant']!r})")
        if uname and (vision.get("sender") or vision.get("receiver")):
            if names_likely_match(vision.get("receiver", ""), uname):
                diagnostics.append(f"INFO [Stage 4]: receiver ตรงกับ USER_NAME ({uname!r}) จึงจัดเป็น 'income'")
            elif names_likely_match(vision.get("sender", ""), uname):
                diagnostics.append(f"INFO [Stage 4]: sender ตรงกับ USER_NAME ({uname!r}) จึงจัดเป็น 'expense'")
        if merged and merged["amount"] == 0:
            diagnostics.append("WARN [Merge]: ยอดเงินสุดท้ายเป็น 0 (อาจเป็นป้าย QR ไม่ระบุยอด หรือ Vision อ่านยอดพลาด)")
        if merged and merged["usedCategoryFallback"]:
            diagnostics.append(f"WARN [Merge]: categoryId จาก LLM ({raw_p.get('categoryId')!r}) ใช้ไม่ได้ จึง fallback เป็น guessCategoryId -> {merged['categoryId']!r}")

    return {
        "filename": filename,
        "stage1_decode": {k: v for k, v in stage1.items() if k != "image_base64"},
        "stage2_tlv": stage2,
        "stage3_llm": stage3,
        "stage4_parse": stage4_parse,
        "final_card": merged,
        "diagnostics": diagnostics,
    }

In [ ]:
from IPython.display import display
from pathlib import Path
from PIL import Image as PILImage
import io as _io

SAMPLES_DIR = REPO_ROOT / "tests" / "llm" / "samples"
IMAGE_PATHS: list[str] = []  # เพิ่ม path นอกโฟลเดอร์ samples ได้ถ้าต้องการ
TEST_IMAGES: dict[str, bytes] = {}

if IN_COLAB:
    from google.colab import files
    print("เลือกไฟล์รูป QR / สลิปที่ต้องการทดสอบ:")
    TEST_IMAGES.update(files.upload())
else:
    if SAMPLES_DIR.is_dir():
        for p in sorted(
            (f for f in SAMPLES_DIR.iterdir() if f.is_file() and f.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}),
            key=lambda f: f.name.lower(),
        ):
            TEST_IMAGES[p.name] = p.read_bytes()
    for raw_p in IMAGE_PATHS:
        path = Path(raw_p)
        if path.is_file():
            TEST_IMAGES[path.name] = path.read_bytes()

if not TEST_IMAGES:
    print(f"ยังไม่มีรูปภาพใน {SAMPLES_DIR} — วางไฟล์รูป .jpg/.png/.webp แล้วรัน Cell นี้อีกครั้ง")
else:
    print(f"โหลดรูปภาพทดสอบทั้งหมด {len(TEST_IMAGES)} ไฟล์: {list(TEST_IMAGES.keys())}")
    LAST_RESULTS = []
    for fname, img_bytes in TEST_IMAGES.items():
        print("=" * 80)
        print(f"FILE: {fname}")
        thumb = PILImage.open(_io.BytesIO(img_bytes))
        thumb.thumbnail((280, 280))
        display(thumb)

        res = run_full_pipeline(img_bytes, fname)
        LAST_RESULTS.append(res)

        s1 = res["stage1_decode"]
        print(f"\n[Stage 1 — QR Decode] size={s1['width']}x{s1['height']} | bytes={s1['file_bytes']} | sent_direct={s1['sent_direct']}")
        print(f"  matched_pass={s1['matched_pass']} | qr_raw={s1['qr_raw']}")

        print(f"\n[Stage 2 — EMVCo TLV Parse]")
        print(" ", json.dumps(res["stage2_tlv"], ensure_ascii=False))

        s3 = res["stage3_llm"]
        print(f"\n[Stage 3 — Vision LLM] ok={s3.get('ok')} | model={s3.get('resolved_model')} | latency={s3.get('latency_ms')}ms | tokens={s3.get('total_tokens')}")
        if s3.get("ok"):
            print("  raw_text:", s3.get("text"))
        else:
            print("  error:", s3.get("error"))

        print(f"\n[Stage 4 — Finora Merged Card]")
        print(" ", json.dumps(res["final_card"], ensure_ascii=False, indent=2))

        print("\n[Diagnostics Checklist]")
        for item in res["diagnostics"]:
            print(" -", item)

In [ ]:
import pandas as pd

# วางโมเดลที่ใช้อยู่ปัจจุบันใน .env.local ไว้ตัวแรก แล้วเทียบกับโมเดล Vision ตัวอื่น
MODELS_TO_COMPARE = list(dict.fromkeys([
    MODEL,
    "google/gemini-2.5-flash",
    "openai/gpt-4o-mini",
]))
DETAILS_TO_COMPARE = ["high"]  # ลองเปลี่ยนเป็น ["low", "high"] เพื่อเทียบผลจาก detail

if not globals().get("TEST_IMAGES"):
    print("กรุณารัน Cell 4 เพื่อโหลดรูปภาพก่อน")
else:
    rows = []
    for fname, img_bytes in TEST_IMAGES.items():
        for mdl in MODELS_TO_COMPARE:
            for dtl in DETAILS_TO_COMPARE:
                res = run_full_pipeline(img_bytes, fname, model=mdl, detail=dtl)
                card = res.get("final_card") or {}
                s3 = res.get("stage3_llm") or {}
                rows.append({
                    "file": fname,
                    "model": mdl,
                    "detail": dtl,
                    "merchant": card.get("merchant", ""),
                    "sender": card.get("sender", ""),
                    "receiver": card.get("receiver", ""),
                    "amount": card.get("amount", ""),
                    "type": card.get("type", ""),
                    "categoryId": card.get("categoryId", ""),
                    "latency_ms": s3.get("latency_ms"),
                    "total_tokens": s3.get("total_tokens"),
                    "error": s3.get("error", ""),
                })

    pd.set_option("display.max_colwidth", None)
    df_compare = pd.DataFrame(rows)
    display(df_compare)

In [ ]:
import pandas as pd

# กรอกเฉลย (Ground Truth) ตามชื่อไฟล์ใน tests/llm/samples/ เพื่อวัดความแม่นยำเวลาปรับ Prompt หรือเปลี่ยนโมเดล
GROUND_TRUTH: dict[str, dict[str, Any]] = {
    "sample_kplus_slip.jpg": {
        "merchant": "Potato shake & ข้าวหมูกรอบ",
        "amount": 30.0,
        "type": "expense",
        "categoryId": "food",
    },
}

# สามารถลองแก้ไข Prompt แล้วส่งผ่าน custom_prompt_fn ได้ (ค่า None = ใช้ build_qr_slip_prompt ของ Finora)
CUSTOM_PROMPT_FN = None

if not GROUND_TRUTH:
    print("ยังไม่ได้กรอก GROUND_TRUTH — ใส่เฉลยตามชื่อไฟล์ด้านบนแล้วรันใหม่เพื่อคิดคะแนนความแม่นยำ")
elif not globals().get("TEST_IMAGES"):
    print("ยังไม่มีรูปใน TEST_IMAGES — กรุณารัน Cell 4 ก่อน")
else:
    eval_rows = []
    for fname, truth in GROUND_TRUTH.items():
        if fname not in TEST_IMAGES:
            print(f"SKIP: ไม่พบไฟล์ {fname!r} ใน TEST_IMAGES")
            continue
        res = run_full_pipeline(TEST_IMAGES[fname], fname, custom_prompt_fn=CUSTOM_PROMPT_FN)
        pred = res.get("final_card") or {}

        exp_m = str(truth.get("merchant", "")).strip()
        got_m = str(pred.get("merchant", "")).strip()
        merchant_exact = got_m == exp_m
        merchant_partial = bool(exp_m and got_m and (exp_m.lower() in got_m.lower() or got_m.lower() in exp_m.lower()))
        amount_ok = abs(float(pred.get("amount") or 0) - float(truth.get("amount") or 0)) < 0.01
        type_ok = pred.get("type") == truth.get("type")
        cat_ok = pred.get("categoryId") == truth.get("categoryId")

        eval_rows.append({
            "file": fname,
            "expected_merchant": exp_m,
            "got_merchant": got_m,
            "merchant_exact": merchant_exact,
            "merchant_partial": merchant_partial,
            "amount_ok": amount_ok,
            "type_ok": type_ok,
            "category_ok": cat_ok,
        })

    if eval_rows:
        df_eval = pd.DataFrame(eval_rows)
        display(df_eval)
        print("\nAccuracy Summary:")
        for col in ("merchant_exact", "merchant_partial", "amount_ok", "type_ok", "category_ok"):
            print(f"  {col}: {df_eval[col].mean() * 100:.1f}% ({int(df_eval[col].sum())}/{len(df_eval)})")